# 4. Evaluation

**Нужны:** все файлы из тетрадок 1–3

**Создаёт:** `evaluation_results.csv`

In [1]:
import csv, math, os, random
from collections import defaultdict, Counter

random.seed(42)
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def parse_vector(s):
    if not s or not s.strip(): return None
    return [float(x) for x in s.split(",")]

def parse_set(s):
    if not s or not s.strip(): return None
    return set(s.split("|"))

def parse_bool(s):
    return str(s).strip().lower() in ("true","1","yes")

DOMAINS = [r["domain_name"].strip()
           for r in load_csv(os.path.join(BASE_DIR,"ontology_domains.csv"))]

SIM_MATRIX = {}
with open(os.path.join(BASE_DIR,"ontology_domain_similarity.csv"),
          newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        d1 = row["domain"].strip()
        for d2 in DOMAINS:
            SIM_MATRIX[(d1,d2)] = float(row.get(d2,0))

ratings_by_id = {r["mentor_id"]: float(r["rating"])
                 for r in load_csv(os.path.join(BASE_DIR,"mentor_ratings.csv"))}

print(f"Областей: {len(DOMAINS)}  |  Рейтингов: {len(ratings_by_id)}")


Областей: 10  |  Рейтингов: 2000


In [2]:
def parse_mentee(r):
    return {
        "id":            r["id"],
        "name":          r["name"],
        "level_score":   int(r["level_score"]),
        "skills_missing": parse_bool(r["skills_missing"]),
        "language_set":  parse_set(r["language_set"]),
        "format_set":    parse_set(r["format_set"]),
        "domain_vector": parse_vector(r["domain_vector"]),
        "skills_list":   [s.strip() for s in
                          r.get("skills_normalized","").split(";") if s.strip()],
    }

def parse_mentor(r):
    return {
        "id":               r["id"],
        "name":             r["name"],
        "profession":       r["profession"],
        "domain":           r.get("domain",""),
        "level_score":      int(r["level_score"]),
        "level_raw":        r.get("level_raw",""),
        "language_set":     parse_set(r["language_set"]),
        "format_set":       parse_set(r["format_set"]),
        "domain_vector":    parse_vector(r["domain_vector"]),
        "skills_list":      [s.strip() for s in
                             r.get("skills_normalized","").split(";") if s.strip()],
        "experience_norm":  float(r["experience_norm"]),
        "experience_years": int(r.get("experience_years",0) or 0),
        "available":        parse_bool(r["available"]),
        "boosted":          parse_bool(r["boosted"]),
        "boost_k":          float(r["boost_k"]),
        "rating":           ratings_by_id.get(r["id"], 3.8),
    }

mentees = [parse_mentee(r)
           for r in load_csv(os.path.join(BASE_DIR,"mentees_processed.csv"))]
mentors = [parse_mentor(r)
           for r in load_csv(os.path.join(BASE_DIR,"mentors_processed.csv"))]

print(f"Mentees: {len(mentees)}  |  Mentors: {len(mentors)}")
print(f"Доступных менторов: {sum(1 for m in mentors if m['available'])}")


Mentees: 5000  |  Mentors: 2000
Доступных менторов: 1613


In [3]:
# ── 4 фактора: skill, domain, exp, rating (goal убран — вклад <0.5%) ─────────

def jaccard(s1, s2):
    a, b = set(s1), set(s2)
    if not a or not b: return None
    return len(a & b) / len(a | b)

def domain_sim_raw(mentee, mentor):
    v1, v2 = mentee["domain_vector"], mentor["domain_vector"]
    if not v1 or not v2: return 0.0
    return sum(
        v1[i] * SIM_MATRIX.get((d1,d2), 0) * v2[j]
        for i,d1 in enumerate(DOMAINS)
        for j,d2 in enumerate(DOMAINS)
    )

def sets_compat(s1, s2):
    if s1 is None or s2 is None: return True
    return len(s1 & s2) > 0

def hard_filter(mentee, pool):
    return [m for m in pool
            if m["available"]
            and m["level_score"] > mentee["level_score"]
            and sets_compat(mentee["language_set"], m["language_set"])
            and sets_compat(mentee["format_set"],   m["format_set"])]

def minmax(v, vmin, vmax):
    if vmax == vmin: return 0.5
    return round(max(0.0, min(1.0, (v - vmin) / (vmax - vmin))), 4)

def compute_score(mentee, mentor, weights):
    # skill: Jaccard. Пустой профиль → 0 (наказание, не исключение)
    sk_raw = jaccard(mentee["skills_list"], mentor["skills_list"])
    do_raw = domain_sim_raw(mentee, mentor)

    skill_val  = minmax(sk_raw, SKILL_MIN, SKILL_MAX) if sk_raw is not None else 0.0
    domain_val = minmax(do_raw, DOMAIN_MIN, DOMAIN_MAX)
    exp_val    = minmax(mentor["experience_norm"], EXP_MIN, EXP_MAX)
    rating_val = minmax((mentor["rating"] - 1) / 4, RATING_MIN, RATING_MAX)

    w_sk = float(weights["w_skills"])
    w_do = float(weights["w_domain"])
    w_ex = float(weights["w_exp"])
    w_ra = float(weights["w_rating"])

    score = w_sk*skill_val + w_do*domain_val + w_ex*exp_val + w_ra*rating_val

    breakdown = {
        "skill":  {"sim":skill_val,  "weight":w_sk,
                   "contribution":round(w_sk*skill_val,4),
                   "penalized": sk_raw is None},
        "domain": {"sim":domain_val, "weight":w_do,
                   "contribution":round(w_do*domain_val,4), "penalized":False},
        "exp":    {"sim":exp_val,    "weight":w_ex,
                   "contribution":round(w_ex*exp_val,4),    "penalized":False},
        "rating": {"sim":rating_val, "weight":w_ra,
                   "contribution":round(w_ra*rating_val,4), "penalized":False},
    }
    return round(score, 4), breakdown

BOOST_THRESHOLD = 0.30
TOP_K = 5

def apply_boost(score, mentor):
    if mentor["boosted"] and score >= BOOST_THRESHOLD:
        return round(score * (1 + mentor["boost_k"]), 4), True
    return score, False

print("Функции скоринга определены (4 фактора: skill, domain, exp, rating)")


Функции скоринга определены (4 фактора: skill, domain, exp, rating)


In [4]:
bounds = {r["factor"]:r for r in load_csv(os.path.join(BASE_DIR,"normalization_bounds.csv"))}
SKILL_MIN,  SKILL_MAX  = float(bounds["skill"]["min"]),  float(bounds["skill"]["max"])
DOMAIN_MIN, DOMAIN_MAX = float(bounds["domain"]["min"]), float(bounds["domain"]["max"])
EXP_MIN,    EXP_MAX    = float(bounds["exp"]["min"]),    float(bounds["exp"]["max"])
RATING_MIN, RATING_MAX = float(bounds["rating"]["min"]), float(bounds["rating"]["max"])
weights_by_id = {w["mentee_id"]:w
                 for w in load_csv(os.path.join(BASE_DIR,"mentee_weights.csv"))}
print("Границы и веса загружены")


Границы и веса загружены


In [5]:
BASE_W = {"w_skills":0.25,"w_domain":0.25,"w_exp":0.25,"w_rating":0.25}
random.seed(99)

def get_top_k(mentee, pool, weights, k=5):
    cands = hard_filter(mentee, pool)
    if not cands: return []
    scored = [(m["id"], apply_boost(compute_score(mentee,m,weights)[0],m)[0]) for m in cands]
    scored.sort(key=lambda x:-x[1])
    return [mid for mid,_ in scored[:k]]

N_TEST = 150
test_mentees = random.sample([m for m in mentees if not m["skills_missing"]], min(N_TEST,len(mentees)))

def make_ideal(mentee, idx):
    return {"id":f"ideal_{idx}","name":f"Ideal {idx}","profession":"","domain":"",
            "level_score":min(mentee["level_score"]+1,4),"level_raw":"senior",
            "language_set":mentee["language_set"] or {"ru"},
            "format_set":mentee["format_set"] or {"online"},
            "domain_vector":mentee["domain_vector"],
            "skills_list":mentee["skills_list"],
            "experience_norm":0.85,"available":True,"boosted":False,"boost_k":0.0,"rating":4.9}

eval_results = []
for i, mentee in enumerate(test_mentees):
    ideal = make_ideal(mentee, i)
    pool  = mentors + [ideal]
    w     = weights_by_id.get(mentee["id"], BASE_W)
    top_k = get_top_k(mentee, pool, w, k=5)
    rank  = top_k.index(ideal["id"])+1 if ideal["id"] in top_k else None
    eval_results.append({"mentee_id":mentee["id"],"found":rank is not None,
                          "ideal_rank":rank,"top_k":"|".join(top_k)})

def recall_at_k(results, k):
    return round(sum(1 for r in results if r["found"] and r["ideal_rank"]<=k)/len(results),4)
def mrr(results):
    return round(sum(1/r["ideal_rank"] for r in results if r["found"])/len(results),4)

all_recs_csv = load_csv(os.path.join(BASE_DIR,"recommendations.csv"))
coverage = round(len({r["mentor_id"] for r in all_recs_csv})/len(mentors),4)

print("="*50)
print("  МЕТРИКИ КАЧЕСТВА")
print("="*50)
print(f"  Recall@1:  {recall_at_k(eval_results,1):.4f}  ({recall_at_k(eval_results,1)*100:.1f}%)")
print(f"  Recall@3:  {recall_at_k(eval_results,3):.4f}  ({recall_at_k(eval_results,3)*100:.1f}%)")
print(f"  Recall@5:  {recall_at_k(eval_results,5):.4f}  ({recall_at_k(eval_results,5)*100:.1f}%)")
print(f"  MRR:       {mrr(eval_results):.4f}")
print(f"  Coverage:  {coverage:.4f}  ({coverage*100:.1f}%)")
print("="*50)

# Стресс-тесты
empty = {"id":"stress","name":"Empty","level_score":1,"skills_missing":True,
         "language_set":None,"format_set":None,
         "domain_vector":[0.1]*len(DOMAINS),"skills_list":[]}
top = get_top_k(empty, mentors, BASE_W)
print(f"\nСтресс-тест (пустой профиль): {len(top)} менторов найдено  {'OK' if top else 'FAIL'}")

with open(os.path.join(BASE_DIR,"evaluation_results.csv"),"w",newline="",encoding="utf-8") as f:
    wcsv = csv.DictWriter(f, fieldnames=["mentee_id","found","ideal_rank","top_k"])
    wcsv.writeheader()
    [wcsv.writerow({**r,"ideal_rank":r["ideal_rank"] or "not_found"}) for r in eval_results]
print(f"evaluation_results.csv создан  ({len(eval_results)} пар)")
print("Тетрадка 4 завершена!")


  МЕТРИКИ КАЧЕСТВА
  Recall@1:  0.2000  (20.0%)
  Recall@3:  0.5533  (55.3%)
  Recall@5:  0.8267  (82.7%)
  MRR:       0.4151
  Coverage:  0.2430  (24.3%)

Стресс-тест (пустой профиль): 5 менторов найдено  OK
evaluation_results.csv создан  (150 пар)
Тетрадка 4 завершена!
